<a href="https://colab.research.google.com/github/gusti-amber/rag/blob/main/chapter5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 5.1 RunnableとRunnableSequence─LCELの最も基本的な構成要素

LCELの最も基本的な実装は、Prompt template・Chat model・Output parserの3つを連結することです。

前章で解説したように、Prompt template、Chat model、Output parserはすべて「invoke」メソッドで実行することができます。

LCELをよく理解できるよう、まずはこれらを順にinvokeすることを試してみます。

まず、prompt（ChatPromptTemplate）・model（ChatOpenAI）・output_parser（StrOutputParser）を準備します。

In [1]:
!pip install --upgrade openai langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.28.0
    Uninstalling openai-2.28.0:
      Successfully uninstalled openai-2.28.0


In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agent-book"

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
  [
      ("system", "ユーザーが入力した料理のレシピを考えてください。"),
      ("human", "{dish}"),
  ]
)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

output_parser = StrOutputParser()

In [ ]:
prompt_value = prompt.invoke({"dish": "カレー"})
ai_message = model.invoke(prompt_value)
output = output_parser.invoke(ai_message)

print(output)

カレーのレシピをご紹介します。シンプルで美味しい基本のカレーを作りましょう。

### 材料（4人分）
- 鶏肉（もも肉または胸肉）: 400g
- 玉ねぎ: 2個
- にんじん: 1本
- じゃがいも: 2個
- カレールー: 1箱（約200g）
- サラダ油: 大さじ2
- 水: 800ml
- 塩: 適量
- 胡椒: 適量
- お好みでガーリックパウダーや生姜: 適量

### 作り方
1. **材料の下ごしらえ**:
   - 鶏肉は一口大に切り、塩と胡椒をふっておきます。
   - 玉ねぎは薄切り、にんじんは輪切り、じゃがいもは一口大に切ります。

2. **炒める**:
   - 大きめの鍋にサラダ油を熱し、玉ねぎを中火で炒めます。玉ねぎが透明になるまで炒めます。
   - 鶏肉を加え、表面が白くなるまで炒めます。

3. **野菜を加える**:
   - にんじんとじゃがいもを鍋に加え、全体をよく混ぜます。

4. **煮る**:
   - 水を加え、強火で煮立たせます。煮立ったら、アクを取り除き、中火にして蓋をし、約15分煮ます。

5. **カレールーを加える**:
   - カレールーを割り入れ、よく溶かします。さらに10分ほど煮込み、全体がなじんだら味を見て、必要に応じて塩や胡椒で調整します。

6. **仕上げ**:
   - お好みでガーリックパウダーや生姜を加えて風味をアップさせます。火を止めて、少し冷ますと味がなじみます。

7. **盛り付け**:
   - ご飯と一緒に盛り付けて、お好みで福神漬けやらっきょうを添えて完成です。

### おすすめのトッピング
- 煮卵
- チーズ
- パクチー

この基本のカレーはアレンジがしやすいので、野菜や肉を変えて楽しんでください！


このようなコードでもPrompt template、Chat model、Output parserを順に実行できますが、LCELではこれらを「|」で連結してから実行します。

実は、ChatPromptTemplate・ChatOpenAI・StrOutputParserは、すべてLangChainの「Runnable」という抽象基底クラスを継承しています。
Runnableを「|」でつなぐと「RunnableSequence」になります。
RunnableSequenceもRunnableの一種です。

In [ ]:
chain = prompt | model | output_parser
output = chain.invoke({"dish": "カレー"})

このように、Runnableを「|」でつないで新たなRunnableを作り、それをinvokeしたときに内部のRunnableが順にinvokeされるというのがLCELの基本です。

### Runnableの実行方法─invoke・stream・batch

### LCELの「|」でさまざまなRunnableを連鎖させる

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

output_parser = StrOutputParser()

In [ ]:
cot_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザーの質問にステップバイステップで回答してください。"),
        ("human", "{question}"),
    ]
)

cot_chain = cot_prompt | model | output_parser

In [ ]:
summarize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ステップバイステップで考えた回答から結論だけ抽出してください"),
        ("human", "{text}"),
    ]
)
summarize_chain = summarize_prompt | model | output_parser

In [ ]:
cot_summarize_chain = cot_chain | summarize_chain
cot_summarize_chain.invoke({"question": "10 + 2 * 3"})

'10 + 2 * 3 の答えは **16** です。'

最終的に要約されたシンプルな回答が得られました。このときcot_summarize_chainの内部では、まずcot_chainが実行され、ステップバイステップで考えた冗長な回答を得ます。その回答を入力としてsummarize_chainを実行し、要約されたシンプルな回答を得ています。LLMを2回呼び出すことで、Zero-shot CoTを使って回答の精度を高めつつ、最終的にはシンプルな出力を得ることができたということです。

## 5.2 RunnableLambda─任意の関数をRunnableにする
LCELでは、Runnable同士を「|」で接続できる以外にも幅広いカスタマイズ方法が提供されています。
ここでは任意の関数をRunnableにする「RunnableLambda」を解説します。

LLMアプリケーションでは、LLMの応答に対してルールベースでさらに処理を加えたり、何らかの変換をかけたいことも多いです。

RunnableLambdaを使うと、LCELのChainに任意の処理（関数）を連結することができます。
例として、LLMの生成したテキストに対して、小文字を大文字に変換する処理を連鎖させるChainを実装してみます。

まず、prompt（ChatPromptTemplate）・model（ChatOpenAI）・output_parser（StrOutputParser）を準備します。

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{input}"),
    ]
)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

output_parser = StrOutputParser()

In [4]:
from langchain_core.runnables import RunnableLambda

def upper(text: str) -> str:
    return text.upper()

chain = prompt | model | output_parser | RunnableLambda(upper)

output = chain.invoke({"input": "Hello!"})
print(output)


HELLO! HOW CAN I ASSIST YOU TODAY?


## 5.3 RunnableParallel─複数のRunnableを並列につなげる

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
output_parser = StrOutputParser()

optimistic_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "あなたは楽観主義者です。ユーザーの入力に対して楽観的な意見をください。"),
        ("human", "{topic}"),
    ]
)
optimistic_chain = optimistic_prompt | model | output_parser

pessimistic_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "あなたは悲観主義者です。ユーザーの入力に対して悲観的な意見をください。"),
        ("human", "{topic}"),
    ]
)
pessimistic_chain = pessimistic_prompt | model | output_parser


In [6]:
import pprint
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    {
        "optimistic_opinion": optimistic_chain,
        "pessimistic_opinion": pessimistic_chain,
    }
)

output = parallel_chain.invoke({"topic": "生成AIの進化について"})
pprint.pprint(output)

{'optimistic_opinion': '生成AIの進化は本当に素晴らしいですね！技術が進むことで、私たちの生活がより便利で豊かになる可能性が広がっています。クリエイティブな作業や問題解決の手助けをしてくれるAIが増えてきて、私たちのアイデアを実現する手助けをしてくれるのです。これからも新しい発見や革新が続くでしょうし、私たちの未来はますます明るくなっていくと思います！どんな素晴らしいことが待っているのか、ワクワクしますね！',
 'pessimistic_opinion': '生成AIの進化は確かに目覚ましいものがありますが、その一方で多くの懸念も抱えています。技術が進化することで、私たちの仕事が奪われたり、情報の信頼性が低下したりするリスクが高まっています。さらに、AIが生成するコンテンツが人間の創造性を脅かし、私たちの思考や感情に悪影響を及ぼす可能性もあります。結局のところ、便利さの裏には常にリスクが潜んでいるのです。私たちがこの技術をどのように扱うかが、未来の社会に大きな影響を与えるでしょうが、その結果が必ずしも良い方向に進むとは限りません。'}


楽観的な意見と悲観的な意見が、dictとして出力されました。
このように、複数のRunnableを並列に接続して実行できるのが「RunnableParallel」です。

実際、上記の例optimistic_chainとpessimistic_chainは同時に実行されるため、順番に実行するよりも短い時間で全体の処理が完了します。
RunnableParallelは、実質的にはキーがstrで値がRunnable（またはRunnableに自動変換できる関数など）であるdictです。

### RunnableParallelの出力をRunnableの入力に連結する
RunnableParallelもRunnableの一種なので、Runnableと「|」で連結することができます。RunnableParallelとRunnableを「|」で連結する例として、楽観的意見と悲観的意見を出したうえで、客観的にまとめるChainを組んでみます。

In [7]:
synthesize_prompt = ChatPromptTemplate.from_messages(
  [
      ("system", "あなたは客観的AIです。2つの意見をまとめてください。"),
      ("human", "楽観的意見: {optimistic_opinion}\n悲観的意見: {pessimistic_opinion}"),
  ]
)

synthesize_chain = (
    RunnableParallel(
        {
            "optimistic_opinion": optimistic_chain,
            "pessimistic_opinion": pessimistic_chain,
        }
    )
    | synthesize_prompt
    | model
    | output_parser
)

output = synthesize_chain.invoke({"topic": "生成AIの進化について"})
print(output)


生成AIの進化については、楽観的な意見と悲観的な意見が存在します。楽観的な見方では、生成AIの技術が進化することで、私たちの生活が便利で豊かになる可能性が広がり、クリエイティブな作業や問題解決の支援を通じてアイデアの実現が促進されると期待されています。これにより、未来は明るいものになると考えられています。

一方で、悲観的な見方では、技術の進化には多くの懸念が伴い、仕事の喪失や情報の信頼性の低下といったリスクが高まると指摘されています。また、AIが生成するコンテンツの氾濫がオリジナリティやクリエイティビティの喪失を招く可能性もあり、便利さの裏には代償があることが懸念されています。このように、生成AIの進化には明るい未来の可能性と同時に、慎重な対応が求められる課題が存在しています。


### RunnableLambdaとの組み合わせ─itemgetterを使う例
LCELでは、このitemgetterを使うと便利な場面が多いです。たとえば、次のコードは{"topic": "生成AIの進化について"}からitemgetter("topic")で値を取り出して、ChatPromptTemplateの{topic}の箇所に穴埋めしています。

In [ ]:
from operator import itemgetter

synthesize_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "あなたは客観的AIです。{topic}について2つの意見をまとめてください。",
        ),
        (
            "human",
            "楽観的意見: {optimistic_opinion}\n悲観的意見: {pessimistic_opinion}",
        ),
    ]
)

synthesize_chain = (
    {
        "optimistic_opinion": optimistic_chain,
        "pessimistic_opinion": pessimistic_chain,
        "topic": itemgetter("topic"),
    }
    | synthesize_prompt
    | model
    | output_parser
)

output = synthesize_chain.invoke({"topic": "生成AIの進化について"})
print(output)

In [8]:
import os
from google.colab import userdata

os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [14]:
!pip install --upgrade langchain_community tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に回答してください。

文脈: """
{context}
"""

質問: {question}
''')

model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)


In [18]:
from langchain_community.retrievers import TavilySearchAPIRetriever

retriever = TavilySearchAPIRetriever(k=3) # webで3件検索する


In [19]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

output = chain.invoke("東京の今日の天気は？")
print(output)

東京の今日の天気は、朝までは広く雨が降り、特に南部では通勤通学の時間帯に本降りの雨となる見込みです。昼頃からは内陸を中心に晴れてくるものの、沿岸部は雲が多い天気が続くでしょう。最高気温は19度前後で、日差しの届く内陸では薄着で過ごせるくらいですが、午後からは北風が強まり、夜は風が冷たく感じられるかもしれませんので、重ね着で調節することをおすすめします。
